# Day 22: Containerization: Docker & NVIDIA NIMs
> *100 Days of Inference* | Layer: **Infrastructure** | Book: *Inference Engineering* Ch 7.1

**Prerequisite:** Day 21

## What problem does this solve?

An inference engine has been built or configured. Now it needs to be deployed reliably, repeatedly, and portably across nodes. Containers solve the "works on my machine" problem by packaging the entire runtime — CUDA, Python dependencies, the inference engine — into a single artifact that lands identically on every host.

The inference container ecosystem is uniquely complex: CUDA versions must match the host driver, the PyTorch version determines which kernels are available, and inference engines have their own compatibility matrices. Getting any of this wrong causes subtle performance regressions or hard failures at startup.

## Concept Overview

Containerization packages model weights, runtime, and configuration into a reproducible unit. Docker images for inference include the CUDA runtime, framework (vLLM/TRT-LLM), and model artifacts. NVIDIA NIM (NVIDIA Inference Microservice) is a standardized container format with an OpenAI-compatible API, built-in health checks, Prometheus metrics, and optimized TensorRT-LLM engines.

**Key terms:**
- **Image:** executable package containing OS + dependencies + code
- **Container:** running instance of an image
- **Dockerfile:** instructions for building an image
- **Registry:** Docker Hub or NVIDIA NGC (NVIDIA GPU Cloud) for storing and distributing images

**NVIDIA NGC** hosts pre-built images for every combination of CUDA, cuDNN, and major ML frameworks. Starting from these official images guarantees driver compatibility — the most common cause of "works on my machine" failures in inference.

**Dependency pinning is non-negotiable.** The inference ecosystem moves fast: `transformers==4.56.2` works today, while `transformers>=4.40.0` may break the moment 5.0 ships. Pin exact versions and bump deliberately.

In [1]:
import subprocess
import json
import os

print("Checking container runtime environment...")

# Check Docker
result = subprocess.run(["docker", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"Docker: {result.stdout.strip()}")
else:
    print("Docker not available in this environment")

# Check NVIDIA container toolkit
result = subprocess.run(["nvidia-ctk", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"NVIDIA Container Toolkit: {result.stdout.strip()}")
else:
    print("nvidia-ctk not available (needed for GPU passthrough)")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Checking container runtime environment...
Docker: Docker version 28.5.1, build e180ab8
NVIDIA Container Toolkit: NVIDIA Container Toolkit CLI version 1.19.0
commit: ec7b4e2fa2caecad6d89be4a26029b831fe7503a
PyTorch: 2.11.0+cu130
CUDA: True


## vLLM Dockerfile: Production Template
A complete, production-ready Dockerfile for a vLLM inference server.

In [2]:
vllm_dockerfile = '''# Production vLLM Inference Server
FROM nvcr.io/nvidia/cuda:12.4.1-cudnn-devel-ubuntu22.04

# System deps
RUN apt-get update && apt-get install -y \\
    python3.11 python3-pip git wget curl \\
    && rm -rf /var/lib/apt/lists/*

# Python deps
RUN pip install --no-cache-dir \\
    vllm>=0.20.0 \\
    fastapi uvicorn \\
    prometheus-client \\
    opentelemetry-api opentelemetry-sdk

# Download model at build time (optional; usually mounted at runtime)
# ARG MODEL_ID=meta-llama/Llama-3.1-8B-Instruct
# RUN huggingface-cli download $MODEL_ID

# Copy server wrapper
COPY server.py /app/server.py
WORKDIR /app

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=120s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# Expose OpenAI-compatible port
EXPOSE 8000

ENV MODEL_ID="meta-llama/Llama-3.1-8B-Instruct"
ENV MAX_MODEL_LEN=8192
ENV TENSOR_PARALLEL_SIZE=1
ENV GPU_MEMORY_UTILIZATION=0.95

CMD ["python3", "-m", "vllm.entrypoints.openai.api_server", \\
     "--model", "$MODEL_ID", \\
     "--max-model-len", "$MAX_MODEL_LEN", \\
     "--tensor-parallel-size", "$TENSOR_PARALLEL_SIZE", \\
     "--gpu-memory-utilization", "$GPU_MEMORY_UTILIZATION", \\
     "--host", "0.0.0.0", \\
     "--port", "8000"]
'''
print("vLLM Dockerfile:")
print(vllm_dockerfile)

vLLM Dockerfile:
# Production vLLM Inference Server
FROM nvcr.io/nvidia/cuda:12.4.1-cudnn-devel-ubuntu22.04

# System deps
RUN apt-get update && apt-get install -y \
    python3.11 python3-pip git wget curl \
    && rm -rf /var/lib/apt/lists/*

# Python deps
RUN pip install --no-cache-dir \
    vllm>=0.20.0 \
    fastapi uvicorn \
    prometheus-client \
    opentelemetry-api opentelemetry-sdk

# Download model at build time (optional; usually mounted at runtime)
# ARG MODEL_ID=meta-llama/Llama-3.1-8B-Instruct
# RUN huggingface-cli download $MODEL_ID

# Copy server wrapper
COPY server.py /app/server.py
WORKDIR /app

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=120s --retries=3 \
    CMD curl -f http://localhost:8000/health || exit 1

# Expose OpenAI-compatible port
EXPOSE 8000

ENV MODEL_ID="meta-llama/Llama-3.1-8B-Instruct"
ENV MAX_MODEL_LEN=8192
ENV TENSOR_PARALLEL_SIZE=1
ENV GPU_MEMORY_UTILIZATION=0.95

CMD ["python3", "-m", "vllm.entrypoints.openai.api_ser

## Docker Layer Anatomy: Why Inference Images Are Huge

A typical web service container is 100–500 MB. An inference container is 10–15 GB before model weights even enter the picture. The CUDA toolkit alone is ~3 GB, the matched cuDNN/NCCL stack adds ~800 MB, and PyTorch + CUDA wheels add another ~5.5 GB. Knowing the layer breakdown is what makes layer caching effective: when application code changes but dependencies don't, only the top layer needs to ship to each node.

In [3]:
# Docker layer structure for a vLLM inference container

container_layers = {
    "ubuntu:22.04 (base OS)": 77,
    "CUDA 12.4 toolkit": 3200,
    "cuDNN + NCCL": 800,
    "Python 3.11 + pip": 45,
    "PyTorch 2.4 + CUDA wheels": 5500,
    "transformers + safetensors": 800,
    "vLLM 0.6 (inference engine)": 1200,
    "Application code": 5,
}

total_mb = sum(container_layers.values())

print("Inference Container Layer Analysis")
print(f"{'Layer':<40} {'Size (MB)':>10} {'Cumulative':>12} {'% of total':>12}")
print("-" * 78)

cumulative = 0
for layer, size_mb in container_layers.items():
    cumulative += size_mb
    pct = size_mb / total_mb * 100
    print(f"{layer:<40} {size_mb:>9} MB {cumulative:>10} MB {pct:>11.1f}%")

print("-" * 78)
print(f"{'Total (without model weights)':<40} {total_mb:>9} MB")
print()
print("Model weights are loaded separately at runtime from object storage.")
print("This lets the same image serve multiple model versions and keeps")
print("cold-start data transfer down: when only application code changes,")
print("layer caching means just the top ~5 MB layer ships instead of all 11 GB.")

Inference Container Layer Analysis
Layer                                     Size (MB)   Cumulative   % of total
------------------------------------------------------------------------------
ubuntu:22.04 (base OS)                          77 MB         77 MB         0.7%
CUDA 12.4 toolkit                             3200 MB       3277 MB        27.5%
cuDNN + NCCL                                   800 MB       4077 MB         6.9%
Python 3.11 + pip                               45 MB       4122 MB         0.4%
PyTorch 2.4 + CUDA wheels                     5500 MB       9622 MB        47.3%
transformers + safetensors                     800 MB      10422 MB         6.9%
vLLM 0.6 (inference engine)                   1200 MB      11622 MB        10.3%
Application code                                 5 MB      11627 MB         0.0%
------------------------------------------------------------------------------
Total (without model weights)                11627 MB

Model weights are loaded s

## NVIDIA NIM: Standardized Inference Containers
NIM provides opinionated, pre-built containers with: OpenAI API, TRT-LLM engines, NIM telemetry, and security.

In [4]:
nim_structure = {
    "Base image": "nvcr.io/nvidia/nim/{model}:{version}",
    "Included components": [
        "TensorRT-LLM engine (pre-compiled for target GPU)",
        "Triton Inference Server",
        "OpenAI-compatible REST API (/v1/completions, /v1/chat/completions)",
        "Prometheus metrics endpoint (/metrics)",
        "Health endpoints (/health/live, /health/ready)",
        "NVIDIA GPU Telemetry",
    ],
    "Required env vars": {
        "NGC_API_KEY": "NVIDIA API key for model download",
        "NIM_CACHE_PATH": "/opt/nim/.cache (model weight cache)",
    },
    "Docker run example": ("docker run -d "
                            "--gpus all "
                            "--shm-size=16GB "
                            "-e NGC_API_KEY=$NGC_API_KEY "
                            "-v ~/.cache/nim:/opt/nim/.cache "
                            "-p 8000:8000 "
                            "nvcr.io/nim/meta/llama-3.1-8b-instruct:latest"),
    "API compatibility": "100% OpenAI-compatible (drop-in replacement)",
}

for key, val in nim_structure.items():
    print(f"\n{key}:")
    if isinstance(val, list):
        for item in val: print(f"  - {item}")
    elif isinstance(val, dict):
        for k, v in val.items(): print(f"  {k}: {v}")
    else:
        print(f"  {val}")


Base image:
  nvcr.io/nvidia/nim/{model}:{version}

Included components:
  - TensorRT-LLM engine (pre-compiled for target GPU)
  - Triton Inference Server
  - OpenAI-compatible REST API (/v1/completions, /v1/chat/completions)
  - Prometheus metrics endpoint (/metrics)
  - Health endpoints (/health/live, /health/ready)
  - NVIDIA GPU Telemetry

Required env vars:
  NGC_API_KEY: NVIDIA API key for model download
  NIM_CACHE_PATH: /opt/nim/.cache (model weight cache)

Docker run example:
  docker run -d --gpus all --shm-size=16GB -e NGC_API_KEY=$NGC_API_KEY -v ~/.cache/nim:/opt/nim/.cache -p 8000:8000 nvcr.io/nim/meta/llama-3.1-8b-instruct:latest

API compatibility:
  100% OpenAI-compatible (drop-in replacement)


## NIM vs Custom Container: When to Reach for Each

NIMs shine for popular models on supported NVIDIA GPUs when deployment speed matters. They lose value when custom CUDA kernels, custom batching policies, or non-standard models are required — the opinionated NIM configuration becomes a constraint rather than a head start. The decision below scores both paths against a few real factors.

In [5]:
# NIM vs custom container decision tree

def nim_or_custom(model_name, gpu_type, need_custom_kernel, need_custom_batching,
                  team_expertise, time_to_deploy_days):
    """Score NIM vs custom container based on requirements."""
    score_nim = 0
    score_custom = 0

    # Factors favoring NIM
    popular_models = ["Llama", "Mistral", "Phi", "Gemma"]
    if any(m in model_name for m in popular_models):
        score_nim += 2  # NIM likely available and well-tested

    nvidia_gpus = ["H100", "A100", "B200"]
    if any(g in gpu_type for g in nvidia_gpus):
        score_nim += 1  # NIM optimized for NVIDIA hardware

    if time_to_deploy_days < 7:
        score_nim += 2  # NIM much faster to deploy

    # Factors favoring custom
    if need_custom_kernel:
        score_custom += 3  # Custom kernels can't slot into a NIM
    if need_custom_batching:
        score_custom += 2  # NIM batching policy is fixed
    if team_expertise > 3:
        score_custom += 1  # Experienced team can handle custom build
    if time_to_deploy_days > 30:
        score_custom += 1  # Time available to build properly

    return "NIM" if score_nim >= score_custom else "Custom", score_nim, score_custom

scenarios = [
    ("Llama 3 70B on H100, quick PoC",
     "Llama 3 70B", "H100", False, False, 2, 3),
    ("DeepSeek + custom FP8 kernel",
     "DeepSeek V3", "H100", True, False, 5, 14),
    ("Custom model, proprietary batching",
     "CustomBERT-v2", "A100", False, True, 4, 30),
    ("Mistral 7B, no GPU expertise, ship in 2 days",
     "Mistral 7B", "A100", False, False, 1, 2),
]

print("NIM vs Custom Container Recommendation")
print()
for name, model, gpu, custom_k, custom_b, exp, days in scenarios:
    rec, s_nim, s_custom = nim_or_custom(model, gpu, custom_k, custom_b, exp, days)
    print(f"Scenario: {name}")
    print(f"  Recommendation: {rec} (NIM score={s_nim}, Custom score={s_custom})")
    print()

NIM vs Custom Container Recommendation

Scenario: Llama 3 70B on H100, quick PoC
  Recommendation: NIM (NIM score=5, Custom score=0)

Scenario: DeepSeek + custom FP8 kernel
  Recommendation: Custom (NIM score=1, Custom score=4)

Scenario: Custom model, proprietary batching
  Recommendation: Custom (NIM score=1, Custom score=3)

Scenario: Mistral 7B, no GPU expertise, ship in 2 days
  Recommendation: NIM (NIM score=5, Custom score=0)



## Container Networking: Expose + Route
Production inference deployments use a load balancer (Nginx/Envoy) in front of multiple model containers.

In [6]:
nginx_config = '''# nginx.conf for vLLM load balancing
upstream vllm_backend {
    # Multiple model instances across spark-01 and spark-02
    server 192.168.1.76:8000 weight=1;
    server 192.168.1.77:8000 weight=1;

    # Least-connections load balancing
    least_conn;

    # Health check (nginx plus required for active health checks)
    keepalive 64;
}

server {
    listen 80;

    # Increase timeouts for long-running inference
    proxy_read_timeout 300s;
    proxy_connect_timeout 5s;

    location /v1/ {
        proxy_pass http://vllm_backend;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;

        # Enable streaming (SSE)
        proxy_buffering off;
        proxy_cache off;
        proxy_set_header Connection "";
        chunked_transfer_encoding on;
    }

    location /health {
        proxy_pass http://vllm_backend/health;
    }

    location /metrics {
        # Restrict metrics to internal network
        allow 192.168.1.0/24;
        deny all;
        proxy_pass http://vllm_backend/metrics;
    }
}
'''
print("Nginx configuration for vLLM load balancing:")
print(nginx_config)

Nginx configuration for vLLM load balancing:
# nginx.conf for vLLM load balancing
upstream vllm_backend {
    # Multiple model instances across spark-01 and spark-02
    server 192.168.1.76:8000 weight=1;
    server 192.168.1.77:8000 weight=1;

    # Least-connections load balancing
    least_conn;

    # Health check (nginx plus required for active health checks)
    keepalive 64;
}

server {
    listen 80;

    # Increase timeouts for long-running inference
    proxy_read_timeout 300s;
    proxy_connect_timeout 5s;

    location /v1/ {
        proxy_pass http://vllm_backend;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;

        # Enable streaming (SSE)
        proxy_buffering off;
        proxy_cache off;
        proxy_set_header Connection "";
        chunked_transfer_encoding on;
    }

    location /health {
        proxy_pass http://vllm_backend/health;
    }

    location /metrics {
        # Restrict metrics to internal network
        a

## Docker Compose: Home Lab Deployment
A complete docker-compose.yml for the spark-01/02 home lab cluster.

In [7]:
docker_compose = '''version: "3.8"

services:
  vllm-spark01:
    image: vllm/vllm-openai:latest
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    environment:
      - HUGGING_FACE_HUB_TOKEN=${HF_TOKEN}
    volumes:
      - ~/.cache/huggingface:/root/.cache/huggingface
    command: >
      --model meta-llama/Llama-3.1-8B-Instruct
      --max-model-len 8192
      --gpu-memory-utilization 0.95
    ports:
      - "8000:8000"
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 5
      start_period: 120s
    restart: unless-stopped

  nginx:
    image: nginx:alpine
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf:ro
    depends_on:
      vllm-spark01:
        condition: service_healthy
'''
print("Docker Compose for home lab deployment:")
print(docker_compose)

Docker Compose for home lab deployment:
version: "3.8"

services:
  vllm-spark01:
    image: vllm/vllm-openai:latest
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 1
              capabilities: [gpu]
    environment:
      - HUGGING_FACE_HUB_TOKEN=${HF_TOKEN}
    volumes:
      - ~/.cache/huggingface:/root/.cache/huggingface
    command: >
      --model meta-llama/Llama-3.1-8B-Instruct
      --max-model-len 8192
      --gpu-memory-utilization 0.95
    ports:
      - "8000:8000"
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 5
      start_period: 120s
    restart: unless-stopped

  nginx:
    image: nginx:alpine
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf:ro
    depends_on:
      vllm-spark01:
        condition: service_healthy



## Experiments: Try These

**Hands-on**

1. Deploy vLLM on spark-01 using the docker-compose template above — time the container startup and first request.
2. Build a custom NIM-compatible container from scratch with a TRT-LLM engine and measure the startup time difference vs vLLM.
3. Add `prometheus-client` to a vLLM server wrapper and emit `inference_requests_total` and `tokens_generated_total` metrics.

**Analytical**

4. Build a `requirements.txt` for a vLLM server, run `pipdeptree` against it, and identify the three largest packages by installed size.
5. For a 12 GB inference container where only application code (5 MB) changes per deploy, compute the bandwidth saved per deploy via layer caching across a 10-node cluster.
6. Estimate cold-start time for a 70B-class NIM: 11 GB image pull at 1 GB/s plus 140 GB weight load at 3.35 TB/s HBM. Which step dominates, and which would be optimized first?

## Key Takeaways

- Docker is the standard deployment unit for inference: it captures the entire runtime environment (CUDA, framework, model loader) for reproducibility.
- Inference containers run 10–15 GB before model weights — CUDA + cuDNN + PyTorch + the engine dominate. Layer caching is what keeps redeploys fast.
- **Pin exact versions.** The inference ecosystem moves fast enough that unpinned dependencies break in weeks, not months.
- **Never bake model weights into the image.** Load them at runtime from object storage so the same image can serve multiple model versions without rebuilds.
- NVIDIA NIM (NVIDIA Inference Microservice) is NVIDIA's opinionated production container — TRT-LLM engine + OpenAI-compatible API + metrics baked in, optimized per GPU type. Excellent for speed-to-deploy on popular models; limiting when deep customization is required.
- The NVIDIA Container Toolkit (`nvidia-container-toolkit`) enables GPU passthrough into Docker — a prerequisite for any containerized GPU inference.
- Health checks, readiness probes, and graceful shutdown are non-optional in production: a vLLM container takes 1–3 minutes to load a model.
- **What's next:** Day 23 — Autoscaling: Concurrency, Batching & Cold Starts.

## References
- *Inference Engineering* Ch 7.1 — Philip Kiely
- [NVIDIA NIM documentation](https://docs.nvidia.com/nim/)
- [vLLM Docker deployment guide](https://docs.vllm.ai/en/latest/deployment/docker.html)